In [1]:
import os
import sys

PROJECT_DIR = (
    "/work/archcomplab/ssaraf/"
    "Maya Archarological Sites/Model Training/"
    "swin3d_project"
)

os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.insert(
        0,
        PROJECT_DIR,
    )

print("Working directory:", os.getcwd())

Working directory: /work/archcomplab/ssaraf/Maya Archarological Sites/Model Training/swin3d_project


In [2]:
from pathlib import Path

import torch

from models.swin3d_model import Model
from datasets.belize_laz_dataset import create_dataloaders
from evaluation.evaluate_avd import evaluate_best_model_avd

/work/archcomplab/ssaraf/envs/maya_env/lib/python3.10/site-packages/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


In [3]:
_, _, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=1,
    voxel_max=65_000,
    num_workers=4,
)

print("Test batches:", len(test_loader))

Test batches: 80


In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = Model(
    num_classes=3,
    in_channels=4,
).to(device)

print("Device:", device)

Device: cuda


In [5]:
BEST_CHECKPOINT = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_checkpoints/dice_soft_avd/swin3d_dice_soft_avd_best_03.pth"
)

AVD_RESULTS_DIR = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_results/generalized_dice/soft_avd"
)

print("Checkpoint exists:", BEST_CHECKPOINT.exists())
print("Checkpoint:", BEST_CHECKPOINT)
print("Output:", AVD_RESULTS_DIR)

Checkpoint exists: True
Checkpoint: /mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/dice_soft_avd/swin3d_dice_soft_avd_best_03.pth
Output: /mnt/nrdstor/archcomplab/ssaraf/swin3d_results/generalized_dice/soft_avd


In [6]:
sample = test_loader.dataset[0]

print("coord:", sample["coord"].shape)
print("metric_coord:", sample["metric_coord"].shape)
print("feat:", sample["feat"].shape)
print("segment:", sample["segment"].shape)

assert sample["coord"].shape[0] == sample["metric_coord"].shape[0]
assert sample["coord"].shape[0] == sample["feat"].shape[0]
assert sample["coord"].shape[0] == sample["segment"].shape[0]

coord: torch.Size([65000, 3])
metric_coord: torch.Size([65000, 3])
feat: torch.Size([65000, 1])
segment: torch.Size([65000])


avd_output = evaluate_best_model_avd(
    model=model,
    test_loader=test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=AVD_RESULTS_DIR,
    device=device,

    cell_size_m=1.0,
    radii_m=(2.0, 5.0, 10.0),
    min_points_per_cell=1,
    min_evidence_cells=3,
    beta=2.0,

    print_progress=True,
    save_csv=True,
)

In [7]:
from torch.utils.data import DataLoader, Subset

small_test_dataset = Subset(
    test_loader.dataset,
    range(min(80, len(test_loader.dataset))),
)

small_test_loader = DataLoader(
    small_test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=test_loader.collate_fn,
)

print("Tiles in quick test:", len(small_test_dataset))

Tiles in quick test: 80


In [8]:
avd_output = evaluate_best_model_avd(
    model=model,
    test_loader=small_test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=AVD_RESULTS_DIR / "quick_test",
    device=device,

    cell_size_m=1.0,
    radii_m=(5.0,),
    min_points_per_cell=1,
    min_evidence_cells=3,
    beta=2.0,

    print_progress=True,
    save_csv=True,
)

/work/archcomplab/ssaraf/envs/maya_env/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/work/archcomplab/ssaraf/software/Swin3D/Swin3D/modules/swin3d_layers.py:239: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:78.)
  offset = torch.cuda.IntTensor(offset)


Evaluated AVD batch 1/80
Evaluated AVD batch 5/80
Evaluated AVD batch 10/80
Evaluated AVD batch 15/80
Evaluated AVD batch 20/80
Evaluated AVD batch 25/80
Evaluated AVD batch 30/80
Evaluated AVD batch 35/80
Evaluated AVD batch 40/80
Evaluated AVD batch 45/80
Evaluated AVD batch 50/80
Evaluated AVD batch 55/80
Evaluated AVD batch 60/80
Evaluated AVD batch 65/80
Evaluated AVD batch 70/80
Evaluated AVD batch 75/80
Evaluated AVD batch 80/80

AVD metric: cell=1 m, minimum evidence=3 cells, F_beta beta=2
class | radius_m | precision | recall | AVD-F_beta | near/total predicted cells | detected/total features
structure | 5 | 0.3422 | 0.4048 | 0.3905 | 2418/7066 | 34/84
plazuela | 5 | 0.1953 | 1.0000 | 0.5482 | 61833/316661 | 91/91
archaeological_discovery | 5 | 0.2134 | 1.0000 | 0.5756 | 68635/321682 | 90/90
macro AVD-F_beta @ 5 m = 0.5047

Checkpoint epoch: 70


In [9]:
avd_results_df = avd_output[
    "results_df"
]

display(avd_results_df)

,Mode / Class,Radius (m),Grid Cell Size (m),Minimum Points Per Cell,Minimum Evidence Cells,Beta,GT Features,Detected Features,Vicinity Recall,Vicinity Precision,AVD-F2,Predicted Cells,Useful Predicted Cells,False Predicted Cells
0,structure,5.0,1.0,1,3,2.0,84,34,0.404762,0.342202,0.390485,7066,2418,4648
1,plazuela,5.0,1.0,1,3,2.0,91,91,1.000000,0.195266,0.548172,316661,61833,254828
2,archaeological_discovery,5.0,1.0,1,3,2.0,90,90,1.000000,0.213363,0.575583,321682,68635,253047


In [10]:
avd_macro_df = avd_output[
    "macro_df"
]

display(avd_macro_df)

,Mode / Class,Radius (m),Structure AVD-F2,Plazuela AVD-F2,Macro AVD-F2
0,archaeological_macro,5.0,0.390485,0.548172,0.469328


In [11]:
display(
    avd_results_df[
        avd_results_df["Radius (m)"] == 5.0
    ]
)

,Mode / Class,Radius (m),Grid Cell Size (m),Minimum Points Per Cell,Minimum Evidence Cells,Beta,GT Features,Detected Features,Vicinity Recall,Vicinity Precision,AVD-F2,Predicted Cells,Useful Predicted Cells,False Predicted Cells
0,structure,5.0,1.0,1,3,2.0,84,34,0.404762,0.342202,0.390485,7066,2418,4648
1,plazuela,5.0,1.0,1,3,2.0,91,91,1.000000,0.195266,0.548172,316661,61833,254828
2,archaeological_discovery,5.0,1.0,1,3,2.0,90,90,1.000000,0.213363,0.575583,321682,68635,253047


# Fast AVD Before vs After Spatial Filter

This version avoids repeatedly running Swin3D while testing the filter.

Workflow:

1. Run model inference once and cache each test tile's metric coordinates, labels, and raw predictions.
2. Apply the spatial filter to cached predictions.
3. Compute AVD before and after filtering at 5 m.


In [12]:
# ============================================================
# Imports for fast before/after evaluation
# ============================================================

import time
import numpy as np
import pandas as pd

from visualization.vis import spatial_filter_predictions

from archaeological_vicinity_metric import (
    AVDConfig,
    ArchaeologicalVicinityMetric,
    ClassSpec,
)


In [13]:
# ============================================================
# Settings
# ============================================================

FILTER_SETTINGS = {
    "cell_size": 1.0,
    "structure_min_cells": 8,
    "plazuela_min_cells": 15,
    "opening_iterations": 1,
    "closing_iterations": 0,
}

AVD_CELL_SIZE = 1.0
AVD_RADII = (5.0,)
AVD_MIN_POINTS_PER_CELL = 1
AVD_MIN_EVIDENCE_CELLS = 3
AVD_BETA = 2.0

print("Filter settings:", FILTER_SETTINGS)
print("AVD radii:", AVD_RADII)


Filter settings: {'cell_size': 1.0, 'structure_min_cells': 8, 'plazuela_min_cells': 15, 'opening_iterations': 1, 'closing_iterations': 0}
AVD radii: (5.0,)


In [14]:
# ============================================================
# AVD helpers
#
# Project labels:
# 0 = Ground
# 1 = Structure
# 2 = Plazuela
# ============================================================

def make_avd_metric():
    return ArchaeologicalVicinityMetric(
        class_specs=[
            ClassSpec(
                "structure",
                predicted_labels=1,
                target_labels=1,
            ),
            ClassSpec(
                "plazuela",
                predicted_labels=2,
                target_labels=2,
            ),
            ClassSpec(
                "archaeological_discovery",
                predicted_labels=(1, 2),
                target_labels=(1, 2),
            ),
        ],
        config=AVDConfig(
            cell_size_m=AVD_CELL_SIZE,
            radii_m=AVD_RADII,
            min_points_per_cell=AVD_MIN_POINTS_PER_CELL,
            min_evidence_cells=AVD_MIN_EVIDENCE_CELLS,
            beta=AVD_BETA,
            ignore_label=-100,
        ),
    )


def result_to_df(result, version):
    rows = []

    for class_name in [
        "structure",
        "plazuela",
        "archaeological_discovery",
    ]:
        metrics = (
            result["per_class"]
            [class_name]
            [5.0]
        )

        predicted_cells = int(
            metrics["predicted_cells"]
        )

        useful_cells = int(
            metrics[
                "predicted_cells_near_gt"
            ]
        )

        rows.append(
            {
                "Version": version,
                "Mode / Class": class_name,
                "Radius (m)": 5.0,
                "GT Features": int(
                    metrics["gt_features"]
                ),
                "Detected Features": int(
                    metrics[
                        "detected_gt_features"
                    ]
                ),
                "Vicinity Recall": float(
                    metrics["feature_recall"]
                ),
                "Vicinity Precision": float(
                    metrics[
                        "vicinity_precision"
                    ]
                ),
                "AVD-F2": float(
                    metrics["avd_fbeta"]
                ),
                "Predicted Cells": (
                    predicted_cells
                ),
                "Useful Predicted Cells": (
                    useful_cells
                ),
                "False Predicted Cells": (
                    predicted_cells
                    -
                    useful_cells
                ),
            }
        )

    return pd.DataFrame(rows)


In [15]:
# ============================================================
# Load best checkpoint
# ============================================================

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

print(
    "Checkpoint epoch:",
    checkpoint.get(
        "epoch",
        "unknown",
    )
)


/tmp/ipykernel_3043066/3285052617.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


Checkpoint epoch: 70


In [16]:
# ============================================================
# STEP 1
# Run Swin3D inference ONCE and cache every test tile
# ============================================================

cached_tiles = []

inference_start = (
    time.perf_counter()
)

with torch.no_grad():

    for batch_index, batch in enumerate(
        test_loader
    ):

        (
            coords,
            features,
            offset,
            labels,
            paths,
            metric_coords,
        ) = batch

        coords_gpu = coords.to(
            device,
            non_blocking=True,
        ).float()

        features_gpu = features.to(
            device,
            non_blocking=True,
        ).float()

        offset_gpu = offset.to(
            device,
            non_blocking=True,
        ).long()

        logits = model(
            (
                coords_gpu,
                features_gpu,
                offset_gpu,
            )
        )

        if isinstance(
            logits,
            tuple,
        ):
            logits = logits[0]

        predictions = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        labels_np = (
            labels
            .cpu()
            .numpy()
        )

        metric_coords_np = (
            metric_coords
            .cpu()
            .numpy()
        )

        start = 0

        for tile_in_batch, end in enumerate(
            offset.tolist()
        ):

            cached_tiles.append(
                {
                    "coords": (
                        metric_coords_np[
                            start:end
                        ].copy()
                    ),

                    "labels": (
                        labels_np[
                            start:end
                        ].copy()
                    ),

                    "predictions": (
                        predictions[
                            start:end
                        ].copy()
                    ),

                    "path": (
                        paths[
                            tile_in_batch
                        ]
                    ),
                }
            )

            start = end

        if (
            batch_index == 0
            or
            (batch_index + 1) % 10 == 0
            or
            batch_index + 1
            ==
            len(test_loader)
        ):
            print(
                f"Inference batch "
                f"{batch_index + 1}/"
                f"{len(test_loader)}"
            )


inference_seconds = (
    time.perf_counter()
    -
    inference_start
)

print()
print(
    "Cached test tiles:",
    len(cached_tiles),
)

print(
    f"Inference time: "
    f"{inference_seconds:.2f} sec"
)


KeyboardInterrupt: 

In [ ]:
# ============================================================
# STEP 2
# Apply spatial filter to cached predictions
# ============================================================

filter_start = (
    time.perf_counter()
)

total_structure_removed = 0
total_plazuela_removed = 0

total_archaeology_before = 0
total_archaeology_after = 0


for tile_index, tile in enumerate(
    cached_tiles
):

    (
        filtered_predictions,
        filter_info,
    ) = spatial_filter_predictions(
        coords=tile["coords"],
        predictions=tile["predictions"],
        **FILTER_SETTINGS,
        verbose=False,
    )

    tile[
        "filtered_predictions"
    ] = filtered_predictions

    total_structure_removed += (
        filter_info[1][
            "removed_points"
        ]
    )

    total_plazuela_removed += (
        filter_info[2][
            "removed_points"
        ]
    )

    total_archaeology_before += int(
        (
            tile["predictions"]
            !=
            0
        ).sum()
    )

    total_archaeology_after += int(
        (
            filtered_predictions
            !=
            0
        ).sum()
    )

    if (
        tile_index == 0
        or
        (tile_index + 1) % 10 == 0
        or
        tile_index + 1
        ==
        len(cached_tiles)
    ):
        print(
            f"Filtered tile "
            f"{tile_index + 1}/"
            f"{len(cached_tiles)}"
        )


filter_seconds = (
    time.perf_counter()
    -
    filter_start
)

print()
print(
    f"Filtering time: "
    f"{filter_seconds:.2f} sec"
)

print(
    "Structure points removed:",
    f"{total_structure_removed:,}",
)

print(
    "Plazuela points removed:",
    f"{total_plazuela_removed:,}",
)

print(
    "Archaeological predictions before:",
    f"{total_archaeology_before:,}",
)

print(
    "Archaeological predictions after:",
    f"{total_archaeology_after:,}",
)


In [ ]:
# ============================================================
# STEP 3
# Compute AVD BEFORE and AFTER
#
# Uses cached predictions, so NO model inference occurs here.
# ============================================================

avd_before_metric = make_avd_metric()
avd_after_metric = make_avd_metric()

avd_start = (
    time.perf_counter()
)

for tile_index, tile in enumerate(
    cached_tiles
):

    coords = tile["coords"]
    labels = tile["labels"]

    # BEFORE
    avd_before_metric.update(
        xyz_m=coords,
        predicted_labels=(
            tile["predictions"]
        ),
        target_labels=labels,
    )

    # AFTER
    avd_after_metric.update(
        xyz_m=coords,
        predicted_labels=(
            tile[
                "filtered_predictions"
            ]
        ),
        target_labels=labels,
    )

    if (
        tile_index == 0
        or
        (tile_index + 1) % 10 == 0
        or
        tile_index + 1
        ==
        len(cached_tiles)
    ):
        print(
            f"AVD tile "
            f"{tile_index + 1}/"
            f"{len(cached_tiles)}"
        )


before_result = (
    avd_before_metric.compute()
)

after_result = (
    avd_after_metric.compute()
)

avd_seconds = (
    time.perf_counter()
    -
    avd_start
)


print()
print("=" * 65)
print("RUNTIME")
print("=" * 65)

print(
    f"Model inference: "
    f"{inference_seconds:.2f} sec"
)

print(
    f"Spatial filter:  "
    f"{filter_seconds:.2f} sec"
)

print(
    f"AVD evaluation:  "
    f"{avd_seconds:.2f} sec"
)

print(
    f"Total:           "
    f"{inference_seconds + filter_seconds + avd_seconds:.2f} sec"
)


In [ ]:
# ============================================================
# BEFORE vs AFTER results
# ============================================================

before_df = result_to_df(
    before_result,
    "Before filter",
)

after_df = result_to_df(
    after_result,
    "After filter",
)

results_before_after = pd.concat(
    [
        before_df,
        after_df,
    ],
    ignore_index=True,
)

display(
    results_before_after.style.format(
        {
            "Radius (m)": "{:.1f}",
            "Vicinity Recall": "{:.4f}",
            "Vicinity Precision": "{:.4f}",
            "AVD-F2": "{:.4f}",
        }
    )
)


In [ ]:
# ============================================================
# Compact comparison
# ============================================================

comparison_rows = []

for class_name in [
    "structure",
    "plazuela",
    "archaeological_discovery",
]:

    before_row = before_df[
        before_df["Mode / Class"]
        ==
        class_name
    ].iloc[0]

    after_row = after_df[
        after_df["Mode / Class"]
        ==
        class_name
    ].iloc[0]

    comparison_rows.append(
        {
            "Mode / Class":
                class_name,

            "Precision Before":
                before_row[
                    "Vicinity Precision"
                ],

            "Precision After":
                after_row[
                    "Vicinity Precision"
                ],

            "Precision Change":
                after_row[
                    "Vicinity Precision"
                ]
                -
                before_row[
                    "Vicinity Precision"
                ],

            "Recall Before":
                before_row[
                    "Vicinity Recall"
                ],

            "Recall After":
                after_row[
                    "Vicinity Recall"
                ],

            "Recall Change":
                after_row[
                    "Vicinity Recall"
                ]
                -
                before_row[
                    "Vicinity Recall"
                ],

            "AVD-F2 Before":
                before_row[
                    "AVD-F2"
                ],

            "AVD-F2 After":
                after_row[
                    "AVD-F2"
                ],

            "AVD-F2 Change":
                after_row[
                    "AVD-F2"
                ]
                -
                before_row[
                    "AVD-F2"
                ],

            "False Cells Before":
                before_row[
                    "False Predicted Cells"
                ],

            "False Cells After":
                after_row[
                    "False Predicted Cells"
                ],
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)

display(
    comparison_df.style.format(
        {
            "Precision Before": "{:.4f}",
            "Precision After": "{:.4f}",
            "Precision Change": "{:+.4f}",

            "Recall Before": "{:.4f}",
            "Recall After": "{:.4f}",
            "Recall Change": "{:+.4f}",

            "AVD-F2 Before": "{:.4f}",
            "AVD-F2 After": "{:.4f}",
            "AVD-F2 Change": "{:+.4f}",
        }
    )
)


In [ ]:
# ============================================================
# Macro AVD-F2 comparison
# ============================================================

def semantic_macro(
    dataframe
):
    semantic = dataframe[
        dataframe["Mode / Class"].isin(
            [
                "structure",
                "plazuela",
            ]
        )
    ]

    return float(
        semantic[
            "AVD-F2"
        ].mean()
    )


macro_before = semantic_macro(
    before_df
)

macro_after = semantic_macro(
    after_df
)


macro_comparison_df = pd.DataFrame(
    [
        {
            "Version":
                "Before filter",

            "Macro AVD-F2":
                macro_before,
        },

        {
            "Version":
                "After filter",

            "Macro AVD-F2":
                macro_after,
        },
    ]
)

display(
    macro_comparison_df.style.format(
        {
            "Macro AVD-F2":
                "{:.4f}"
        }
    )
)

print(
    "Macro AVD-F2 change:",
    f"{macro_after - macro_before:+.4f}",
)
